The file `amazon-meta.txt` contains product information with these attributes:
- **id**: Numeric identifier
- **ASIN**: Amazon Standard Identification Number
- **title**: Product title
- **group**: Product category (Book, Music, DVD, Video)
- **salesrank**: Sales ranking
- **similar**: Related products (by ASIN) (È Amazon stessa che definisce la similarità, tramite il suo algoritmo interno di "Customers who bought this also bought".)
- **categories**: Product categories
- **reviews**: User reviews with ratings

---

## 1. Exploratory Data Analysis of the Graph

In [1]:
# inizialze Spark Session
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("AmazonGraphAnalysis") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.4-s_2.12") \
    .config("spark.driver.memory", "12g") \
    .config("spark.sql.shuffle.partitions", "60") \
    .getOrCreate()

sc = spark.sparkContext 
sc.setLogLevel("WARN") # reduce log level too avoid too much log output

26/04/20 14:42:45 WARN Utils: Your hostname, MacBook-Air-di-Filippo-5.local resolves to a loopback address: 127.0.0.1; using 10.208.164.11 instead (on interface en0)
26/04/20 14:42:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/filippostanghellini/.ivy2/cache
The jars for the packages stored in: /Users/filippostanghellini/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-da47ce2a-f991-4961-b2a0-c99df596bc31;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.4-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 55ms :: artifacts dl 2ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.4-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modu

:: loading settings :: url = jar:file:/opt/anaconda3/envs/DataMining/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/04/20 14:42:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


**nodes.csv** and **edges_ids.txt** is pre-processed csv from amazon-meta.txt

In [2]:
# Upload data
nodes_df = spark.read.csv("data/nodes.csv", header=True, inferSchema=True)
nodes_df = nodes_df.withColumn("id", F.col("id").cast("integer"))
nodes_df = nodes_df.withColumn("salesrank", F.col("salesrank").cast("integer"))

# Upload edges 
edges_df = spark.read.csv("data/edges_ids.txt", sep="\t", schema="src STRING, dst STRING")
edges_df = edges_df.withColumn("src", F.col("src").cast("integer"))
edges_df = edges_df.withColumn("dst", F.col("dst").cast("integer"))

# Upload ratings
ratings_df = spark.read.csv("data/ratings.csv", header=True, inferSchema=True)
ratings_df = ratings_df.withColumn("product_id", F.col("product_id").cast("integer"))
ratings_df = ratings_df.withColumn("rating", F.col("rating").cast("float"))
ratings_df = ratings_df.na.drop()

In [3]:
# nodes, edges, ratings overview

print("Total nodes:", nodes_df.count())
nodes_df.show(5, truncate=False) # expample of nodes

print("Total edges:", edges_df.count())
edges_df.show(5, truncate=False) # example of edges, where src is the source and dst is the destination of the edge

print("Total ratings:", ratings_df.count())

Total nodes: 548552
+---+----------+------------------------------------------------------------+-----+---------+
|id |asin      |title                                                       |group|salesrank|
+---+----------+------------------------------------------------------------+-----+---------+
|0  |0771044445|null                                                        |null |null     |
|1  |0827229534|Patterns of Preaching: A Sermon Sampler                     |Book |396585   |
|2  |0738700797|Candlemas: Feast of Flames                                  |Book |168596   |
|3  |0486287785|World War II Allied Fighter Planes Trading Cards            |Book |1270652  |
|4  |0842328327|Life Application Bible Commentary: 1 and 2 Timothy and Titus|Book |631289   |
+---+----------+------------------------------------------------------------+-----+---------+
only showing top 5 rows

Total edges: 1231439
+---+------+
|src|dst   |
+---+------+
|1  |161555|
|1  |244916|
|1  |118052|
|1  |44423

Total ratings: 7593244


In [4]:
# groups distribution
group_counts = nodes_df.groupBy("group").count().orderBy(F.desc("count"))
group_counts.show(5, truncate=False)

+-----+------+
|group|count |
+-----+------+
|Book |393363|
|Music|103040|
|Video|26119 |
|DVD  |19823 |
|null |5868  |
+-----+------+
only showing top 5 rows



In [5]:
# out-degree distribution
out_degrees = edges_df.groupBy("src").count().withColumnRenamed("count", "out_degree")

# out-degree distribution statistics (table)
out_degrees.select(
    F.min("out_degree").alias("min"),
    F.max("out_degree").alias("max"),
    F.avg("out_degree").alias("mean"),
    F.median("out_degree").alias("median")
).show()

# products with most similar products (top 10)
out_degrees.orderBy(F.desc("out_degree")).limit(10).show(truncate=False)

# In questo caso non ha senso visualizzare tutti i nomi dei prodotti con out degree 5, nonostante sia il massimo.
# dato che molti prodotti hanno out degree 5.

+---+---+------------------+------+
|min|max|              mean|median|
+---+---+------------------+------+
|  1|  5|3.4058390284511586|   4.0|
+---+---+------------------+------+

+----+----------+
|src |out_degree|
+----+----------+
|1517|5         |
|2659|5         |
|1170|5         |
|4533|5         |
|477 |5         |
|2728|5         |
|4101|5         |
|1096|5         |
|935 |5         |
|2792|5         |
+----+----------+



In [6]:
# in-degree distribution
in_degrees = edges_df.groupBy("dst").count().withColumnRenamed("count", "in_degree")

in_degrees.select(
    F.min("in_degree").alias("min"),
    F.max("in_degree").alias("max"),
    F.avg("in_degree").alias("mean"),
    F.median("in_degree").alias("median")
).show()

# products with most in-degrees (top 10)
in_degrees.orderBy(F.desc("in_degree")).limit(10).show(truncate=False)

# products with most in-degrees (top 10) with names
target_dst = [548091, 458358, 222074, 199628, 515301, 291117, 502784, 296016, 239107, 436020]

table_with_names = (
    in_degrees
    .filter(F.col("dst").isin(target_dst))
    .join(
        nodes_df.select(
            F.col("id").alias("dst"),
            F.col("title").alias("product_name")
        ),
        on="dst",
        how="left"
    )
    .select("dst", "in_degree", "product_name")
    .orderBy(F.desc("in_degree"))
)

table_with_names.show(truncate=False)


+---+---+-----------------+------+
|min|max|             mean|median|
+---+---+-----------------+------+
|  1|549|5.190098075164267|   3.0|
+---+---+-----------------+------+

+------+---------+
|dst   |in_degree|
+------+---------+
|548091|549      |
|458358|324      |
|222074|256      |
|199628|230      |
|515301|228      |
|291117|219      |
|502784|216      |
|296016|212      |
|239107|205      |
|436020|197      |
+------+---------+

+------+---------+---------------------------------------------------------------------------------------------------------------------------------------+
|dst   |in_degree|product_name                                                                                                                           |
+------+---------+---------------------------------------------------------------------------------------------------------------------------------------+
|548091|549      |Laura                                                                     

In [7]:
# Ratings analysis

print("Ratings overview:")
ratings_df.select(
    F.count("rating").alias("count"),
    F.min("rating").alias("min"),
    F.max("rating").alias("max"),
    F.avg("rating").alias("mean"),
    F.stddev("rating").alias("std")
).show()

print("\n=== Distribution of ratings ===")
ratings_df.groupBy("rating").count().orderBy("rating").show(truncate=False)

print("\n=== Users with most reviews (top 10) ===")
ratings_df.groupBy("user_id").count().orderBy(F.desc("count")).limit(10).show(truncate=False)

print("\n=== Products with most reviews (top 10) ===")
ratings_df.groupBy("product_id").count().orderBy(F.desc("count")).limit(10).show(truncate=False)

Ratings overview:
+-------+---+---+-----------------+------------------+
|  count|min|max|             mean|               std|
+-------+---+---+-----------------+------------------+
|7593244|1.0|5.0|4.178371720966691|1.2500677208072952|
+-------+---+---+-----------------+------------------+


=== Distribution of ratings ===


+------+-------+
|rating|count  |
+------+-------+
|1.0   |583766 |
|2.0   |415312 |
|3.0   |627917 |
|4.0   |1401990|
|5.0   |4564259|
+------+-------+


=== Users with most reviews (top 10) ===


+--------------+------+
|user_id       |count |
+--------------+------+
|ATVPDKIKX0DER |945065|
|A3UN6WX5RRO2AG|201770|
|A14OJS0VWMOSWO|9795  |
|A2NJO6YE954DBH|6324  |
|AFVQZQ8PW0L   |5441  |
|A9Q28YTLYREO7 |4296  |
|A1K1JW1C5CUSUZ|3576  |
|AU8552YCOO5QX |2864  |
|A3LZGLA88K0LA0|2276  |
|A20EEWWSFMZ1PN|2181  |
+--------------+------+


=== Products with most reviews (top 10) ===
+----------+-----+
|product_id|count|
+----------+-----+
|91158     |4995 |
|258380    |4995 |
|428073    |4995 |
|23792     |4995 |
|429348    |4995 |
|250879    |4995 |
|211463    |4995 |
|546259    |4995 |
|526761    |4995 |
|6130      |4995 |
+----------+-----+



---

<a id='section-2'></a>
## 2. PageRank con GraphFrames

Calcolo dell'importanza dei nodi nel grafo utilizzando l'algoritmo PageRank.

In [8]:
# GraphFrame creation

from graphframes import GraphFrame

# Assicurati che i nodi abbiano il nome corretto della colonna
vertices_df = nodes_df.withColumnRenamed("id", "id")

# Crea il GraphFrame
g = GraphFrame(vertices_df, edges_df)

print("GraphFrame creato:")
print("  - Vertices:", g.vertices.count())
print("  - Edges:", g.edges.count())

GraphFrame creato:
  - Vertices: 548552
  - Edges: 1231439


In [10]:
# PageRank calculation

# PageRank with reset probability = 0.15 (standard)
pagerank_results = g.pageRank(resetProbability=0.15, maxIter=10)
print("PageRank done")

# Unisci risultati con attributi nodi
pagerank_with_info = pagerank_results.vertices.select(
    "id", "asin", "title", "group", "salesrank", "pagerank"
)

print("\nSchema risultati:")
pagerank_with_info.printSchema()

PageRank done

Schema risultati:
root
 |-- id: integer (nullable = true)
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- group: string (nullable = true)
 |-- salesrank: integer (nullable = true)
 |-- pagerank: double (nullable = true)



Salesrank misura la popolarità assoluta, PageRank misura la centralità nei pattern di co-acquisto relazionale.

In [11]:
# top 20 products by PageRank

print("=== TOP 20 PRODUCTS BY PAGERANK ===")
top_pagerank = pagerank_with_info.orderBy(F.desc("pagerank")).limit(20)
top_pagerank.show(truncate=False)

=== TOP 20 PRODUCTS BY PAGERANK ===
+------+----------+---------------------------------------------------------------------------------------------------------------------------------------+-----+---------+------------------+
|id    |asin      |title                                                                                                                                  |group|salesrank|pagerank          |
+------+----------+---------------------------------------------------------------------------------------------------------------------------------------+-----+---------+------------------+
|154855|0066620996|Good to Great: Why Some Companies Make the Leap... and Others Don't                                                                    |Book |29       |186.9563673383964 |
|98756 |0316769487|The Catcher in the Rye                                                                                                                 |Book |60       |176.1265140160191 |
|458358|0

In [12]:
# top pageranked product per category

print("=== TOP 5 PRODUCTS PER CATEGORY ===")

for group in ["Book", "DVD", "Music", "Video"]:
    print("\n---", group.upper(), "---")
    group_top = pagerank_with_info.filter(F.col("group") == group).orderBy(F.desc("pagerank")).limit(5)
    group_top.show(truncate=False)

=== TOP 5 PRODUCTS PER CATEGORY ===

--- BOOK ---
+------+----------+---------------------------------------------------------------------------------------------------------------------------------------+-----+---------+------------------+
|id    |asin      |title                                                                                                                                  |group|salesrank|pagerank          |
+------+----------+---------------------------------------------------------------------------------------------------------------------------------------+-----+---------+------------------+
|154855|0066620996|Good to Great: Why Some Companies Make the Leap... and Others Don't                                                                    |Book |29       |186.9563673383964 |
|98756 |0316769487|The Catcher in the Rye                                                                                                                 |Book |60       |176.12651401601

In [15]:
# PageRank statistics

print("=== PageRank Statistics ===")
pagerank_with_info.select(
    F.min("pagerank").alias("min"),
    F.max("pagerank").alias("max"),
    F.avg("pagerank").alias("mean"),
    F.median("pagerank").alias("median"),
    F.stddev("pagerank").alias("std")
).show()

# Valori elevati di PageRank indicano nodi importanti (molti link in entrata da nodi importanti)
print("\n(Products with pagerank > 2x mean):")
mean_pr = pagerank_with_info.agg(F.avg("pagerank")).first()[0]
high_pagerank = pagerank_with_info.filter(F.col("pagerank") > 2 * mean_pr)
print("Count:", high_pagerank.count())
high_pagerank.orderBy(F.desc("pagerank")).show(truncate=False)

=== PageRank Statistics ===
+------------------+-----------------+------------------+------------------+------------------+
|               min|              max|              mean|            median|               std|
+------------------+-----------------+------------------+------------------+------------------+
|0.2147629990052919|186.9563673383964|0.9999999999999588|0.2147629990052919|2.4676155901079784|
+------------------+-----------------+------------------+------------------+------------------+


(Products with pagerank > 2x mean):
Count: 68254
+------+----------+---------------------------------------------------------------------------------------------------------------------------------------+-----+---------+------------------+
|id    |asin      |title                                                                                                                                  |group|salesrank|pagerank          |
+------+----------+----------------------------------------

In [22]:
# salesrank vs PageRank 

# correlation
correlation = pagerank_with_info.agg(F.corr("salesrank", "pagerank")).first()[0]
print("Correlation between salesrank and pagerank:", correlation)

# correlation in the top 20 pageranked products
top_20_correlation = top_pagerank.agg(F.corr("salesrank", "pagerank")).first()[0]
print("Correlation in top 20 pageranked products:", top_20_correlation)

Correlation between salesrank and pagerank: -0.21143209249539596
Correlation in top 20 pageranked products: -0.30006041967389296


In [23]:
# select pagerank
pagerank_with_info.select(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("salesrank").isNull(), 1).otherwise(0)).alias("null_salesrank"),
    F.sum(F.when(F.col("salesrank") <= 0, 1).otherwise(0)).alias("non_positive_salesrank")
).show()

# select top 1000 salesrank "clean"
top_1000_salesrank = (
    pagerank_with_info
    .filter(F.col("salesrank").isNotNull() & (F.col("salesrank") > 0))
    .orderBy(F.asc("salesrank"))
    .limit(1000)
    .select("id")
    .distinct()
)

# select top 20 pagerank
top_20_pagerank = (
    pagerank_with_info
    .orderBy(F.desc("pagerank"))
    .limit(20)
    .select("id", "salesrank", "pagerank")
)

# overlap between top 20 pagerank and top 1000 salesrank
overlap = top_20_pagerank.join(top_1000_salesrank, on="id", how="inner")

print("Overlap:", overlap.count(), "out of 20")
overlap.orderBy(F.asc("salesrank")).show(truncate=False)

print("Top 20 pagerank salesrank:")
top_20_pagerank.orderBy(F.asc("salesrank")).show(truncate=False)

+------+--------------+----------------------+
|  rows|null_salesrank|non_positive_salesrank|
+------+--------------+----------------------+
|548552|          6186|                   500|
+------+--------------+----------------------+

Overlap: 13 out of 20
+------+---------+------------------+
|id    |salesrank|pagerank          |
+------+---------+------------------+
|154855|29       |186.9563673383964 |
|62424 |42       |126.34663216811431|
|98756 |60       |176.1265140160191 |
|243404|94       |98.47033978163822 |
|548091|110      |138.8286309269328 |
|537519|111      |119.93164582203606|
|296016|122      |97.46486776430726 |
|341570|126      |97.57686600190657 |
|35512 |143      |126.5229456438615 |
|239107|249      |110.23268732072933|
|335281|250      |97.36617299631592 |
|515301|290      |95.67996520261424 |
|458358|371      |160.38958698605254|
+------+---------+------------------+

Top 20 pagerank salesrank:
+------+---------+------------------+
|id    |salesrank|pagerank    

---

<a id='section-3'></a>
## 3. Recommender System con ALS

Implementazione di un sistema di raccomandazione usando **Alternating Least Squares (ALS)** matrix factorization.

ALS decompose la matrice user-item ratings in due matrici fattoriali:
- **U**: Matrice utente-fattore (latent user factors)
- **V**: Matrice item-fattore (latent item factors)

$R \approx U \times V^T$

In [24]:
# Recommender system with ALS

from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer, IndexToString

# Controlla dati ratings
print("Dati ratings prima della trasformazione:")
ratings_df.select("user_id", "product_id", "rating").show(10, truncate=False)

# Count utenti e prodotti unici
num_users = ratings_df.select("user_id").distinct().count()
num_items = ratings_df.select("product_id").distinct().count()
num_ratings = ratings_df.count()

print("\nStatistiche:")
print("  - Utenti unici:", num_users)
print("  - Prodotti unici:", num_items)
print("  - Totali ratings:", num_ratings)

Dati ratings prima della trasformazione:
+--------------+----------+------+
|user_id       |product_id|rating|
+--------------+----------+------+
|A2JW67OY8U6HHK|1         |5.0   |
|A2VE83MZF98ITY|1         |5.0   |
|A11NCO6YTE4BTJ|2         |5.0   |
|A9CQ3PLRNIR83 |2         |4.0   |
|A13SG9ACZ9O5IM|2         |5.0   |
|A1BDAI6VEYMAZA|2         |5.0   |
|A2P6KAWXJ16234|2         |4.0   |
|AMACWC3M7PQFR |2         |4.0   |
|A3GO7UV9XX14D8|2         |4.0   |
|A1GIL64QK68WKL|2         |5.0   |
+--------------+----------+------+
only showing top 10 rows




Statistiche:
  - Utenti unici: 1555170
  - Prodotti unici: 402724
  - Totali ratings: 7593244


In [26]:
# Indexing IDs for ALS


# Create indexer per user_id
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_idx", handleInvalid="keep")
user_model = user_indexer.fit(ratings_df)
ratings_indexed = user_model.transform(ratings_df)

# Create indexer per product_id
product_indexer = StringIndexer(inputCol="product_id", outputCol="product_idx", handleInvalid="keep")
product_model = product_indexer.fit(ratings_indexed)
ratings_indexed = product_model.transform(ratings_indexed)

# Verifica trasformazione
print("Dati dopo indexing:")
ratings_indexed.select("user_id", "user_idx", "product_id", "product_idx", "rating").show(10)

Dati dopo indexing:


26/04/20 15:04:06 WARN DAGScheduler: Broadcasting large task binary with size 81.3 MiB


+--------------+---------+----------+-----------+------+
|       user_id| user_idx|product_id|product_idx|rating|
+--------------+---------+----------+-----------+------+
|A2JW67OY8U6HHK|1115824.0|         1|   264254.0|   5.0|
|A2VE83MZF98ITY|     74.0|         1|   264254.0|   5.0|
|A11NCO6YTE4BTJ| 166918.0|         2|    93254.0|   5.0|
| A9CQ3PLRNIR83|  11377.0|         2|    93254.0|   4.0|
|A13SG9ACZ9O5IM|  70193.0|         2|    93254.0|   5.0|
|A1BDAI6VEYMAZA| 213866.0|         2|    93254.0|   5.0|
|A2P6KAWXJ16234|   1020.0|         2|    93254.0|   4.0|
| AMACWC3M7PQFR|   3263.0|         2|    93254.0|   4.0|
|A3GO7UV9XX14D8|  10063.0|         2|    93254.0|   4.0|
|A1GIL64QK68WKL|  45479.0|         2|    93254.0|   5.0|
+--------------+---------+----------+-----------+------+
only showing top 10 rows



In [27]:
# Train-Test Split (80-20)

train_data, test_data = ratings_indexed.randomSplit([0.8, 0.2], seed=42)

print("Training set:", train_data.count(), "ratings")
print("Test set:", test_data.count(), "ratings")

# Verifica overlap utenti
train_users = train_data.select("user_idx").distinct().count()
test_users = test_data.select("user_idx").distinct().count()
overlap_users = train_data.select("user_idx").intersect(test_data.select("user_idx")).count()

print("\nUtenti in train:", train_users)
print("Utenti in test:", test_users)
print("Utenti in entrambi:", overlap_users)

26/04/20 15:04:13 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB


Training set: 6074107 ratings


26/04/20 15:04:21 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB


Test set: 1519137 ratings


26/04/20 15:04:29 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:04:37 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:04:43 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:04:51 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:04:55 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:05:01 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:05:13 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:05:19 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB



Utenti in train: 1392375
Utenti in test: 605733
Utenti in entrambi: 442938


In [ ]:
# ALS model tuning

print("Addestramento modello ALS...")
print("(Questo potrebbe richiedere diversi minuti)")

# ALS model configuration
als = ALS(
    maxIter=10, # numero di iterazioni
    regParam=0.1, # parametro di regolarizzazione, evitare overfitting
    rank=10,  # numero di fattori latenti
    userCol="user_idx",
    itemCol="product_idx",
    ratingCol="rating",
    coldStartStrategy="drop",  # drop predictions per nuovi utenti/prodotti
    nonnegative=True,  # ratings non negativi, mantenere stabilità e interpretabilità
    implicitPrefs=False # ratings espliciti, non preferenze implicite (click, view, buy)
)

# Fit model using spark MLlib ALS implementation
als_model = als.fit(train_data)

print("Modello ALS addestrato con successo!")

Addestramento modello ALS...
(Questo potrebbe richiedere diversi minuti)


26/04/20 15:11:10 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:11:14 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:11:21 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:11:28 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:11:34 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:11:38 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:11:42 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:11:45 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/20 15:11:45 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/04/20 15:11:46 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:11:50 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB


Modello ALS addestrato con successo!


In [29]:
# Factor Matrices (U e V)


# User factors (U matrix): num_users x rank
print("=== User Factors Matrix (U) ===")
user_factors = als_model.userFactors
user_factors.printSchema()
print("Shape:", user_factors.count(), "users x", len(user_factors.select("features").first()[0]), "factors")
print("\nEsempio (first 3 users):")
user_factors.select("id", "features").limit(3).show(truncate=False)

# Item factors (V matrix): num_items x rank
print("\n=== Item Factors Matrix (V) ===")
item_factors = als_model.itemFactors
item_factors.printSchema()
print("Shape:", item_factors.count(), "items x", len(item_factors.select("features").first()[0]), "factors")
print("\nEsempio (first 3 items):")
item_factors.select("id", "features").limit(3).show(truncate=False)

=== User Factors Matrix (U) ===
root
 |-- id: integer (nullable = false)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = false)



26/04/20 15:15:45 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:15:48 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB


Shape: 1392375 users x 10 factors

Esempio (first 3 users):


26/04/20 15:15:50 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB


+---+-----------------------------------------------------------------------------------------------------------------+
|id |features                                                                                                         |
+---+-----------------------------------------------------------------------------------------------------------------+
|0  |[0.29674974, 0.42713496, 0.28923264, 0.5860622, 0.5590501, 0.375302, 1.6191211, 0.7249973, 0.34351024, 0.2662246]|
|10 |[0.97146255, 0.64042723, 0.21266317, 0.5851201, 0.6758499, 0.7552396, 1.3032787, 0.5114303, 0.31366, 0.0]        |
|20 |[0.7429306, 0.4942213, 0.815377, 0.4499981, 0.8184382, 0.95340574, 1.0023714, 0.50902057, 0.07260874, 0.0]       |
+---+-----------------------------------------------------------------------------------------------------------------+


=== Item Factors Matrix (V) ===
root
 |-- id: integer (nullable = false)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = false

26/04/20 15:15:51 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:15:54 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB


Shape: 383748 items x 10 factors

Esempio (first 3 items):


26/04/20 15:15:55 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB


+---+------------------------------------------------------------------------------------------------------------------+
|id |features                                                                                                          |
+---+------------------------------------------------------------------------------------------------------------------+
|0  |[0.29974252, 0.5799804, 0.2608894, 0.55914456, 0.5217345, 0.39383522, 1.7641394, 0.8171857, 0.3522296, 0.21693385]|
|10 |[0.28614137, 0.57127684, 0.25831118, 0.56410986, 0.5180861, 0.3884586, 1.7688622, 0.8221633, 0.352421, 0.2318123] |
|20 |[0.77128905, 0.9705334, 0.6860665, 0.49303967, 0.7515192, 0.16767482, 1.0892619, 0.253874, 0.71090394, 0.50246906]|
+---+------------------------------------------------------------------------------------------------------------------+



In [30]:
# Inference on test set

# Make predictions on test set
predictions = als_model.transform(test_data)

print("Predictions sample:")
predictions.select("user_idx", "product_idx", "rating", "prediction").show(10)

# Handle NaN predictions (cold start)
predictions_clean = predictions.na.drop(subset=["prediction"])
print("\nPredictions valide:", predictions_clean.count(), "/", predictions.count())

Predictions sample:


26/04/20 15:16:38 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:16:42 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:16:46 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:16:53 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB
26/04/20 15:17:00 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB


+--------+-----------+------+----------+
|user_idx|product_idx|rating|prediction|
+--------+-----------+------+----------+
|    23.0|        8.0|   5.0|  5.020836|
|   536.0|        8.0|   5.0|  4.502142|
|  1143.0|        8.0|   4.0| 3.7288113|
|  1896.0|        8.0|   3.0| 3.8276262|
|  2763.0|        8.0|   3.0| 3.1520314|
|  6902.0|        8.0|   5.0| 4.7910867|
|  6904.0|        8.0|   5.0| 4.6575537|
|  7718.0|        8.0|   4.0|  4.175553|
| 10862.0|        8.0|   3.0|  4.187926|
| 14026.0|        8.0|   5.0| 4.9892797|
+--------+-----------+------+----------+
only showing top 10 rows



26/04/20 15:17:02 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:17:07 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:17:11 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:17:16 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB
26/04/20 15:17:24 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB
26/04/20 15:17:30 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:17:35 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:17:40 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:17:46 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB
26/04/20 15:17:55 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB



Predictions valide: 1327103 / 1327103


In [ ]:
# Model evaluation (RMSE)

from pyspark.ml.evaluation import RegressionEvaluator

# RMSE evaluator
evaluator = RegressionEvaluator(
    labelCol="rating",
    predictionCol="prediction",
    metricName="rmse"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions_clean)

print("=== Model Evaluation ===")
print("RMSE (Root Mean Squared Error):", rmse)

26/04/20 15:19:08 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:19:13 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:19:17 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/20 15:19:22 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB
26/04/20 15:19:31 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB
26/04/20 15:19:37 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB


=== Model Evaluation ===
RMSE (Root Mean Squared Error): 0.9469860102652424

Interpretazione RMSE:
- RMSE = 0: previsioni perfette
- RMSE < 1.0: buon modello (scala rating 1-5)
- RMSE = 0.9469860102652424 : errore medio di 0.95 punti per rating


In [ ]:
#TODO: check from here

# Generation of recommendations for all users

# This can be time consuming

print("Generazione raccomandazioni per tutti gli utenti...")

# Get top 10 recommendations per utente
user_recs = als_model.recommendForAllUsers(10)

print("\nSchema raccomandazioni:")
user_recs.printSchema()
print("\nEsempio raccomandazioni (first 5 users):")
user_recs.show(5, truncate=False)

Generazione raccomandazioni per tutti gli utenti...

Schema raccomandazioni:
root
 |-- user_idx: integer (nullable = false)
 |-- recommendations: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_idx: integer (nullable = true)
 |    |    |-- rating: float (nullable = true)


Esempio raccomandazioni (first 5 users):


26/04/20 15:23:15 WARN DAGScheduler: Broadcasting large task binary with size 90.8 MiB
ERROR:root:KeyboardInterrupt while sending command.             (20 + 10) / 100]
Traceback (most recent call last):
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# ============================================
# Espansione raccomandazioni con info prodotto
# ============================================

from pyspark.sql.functions import explode, col

# Espandi la colonna recommendations
exploded_recs = user_recs.select(
    col("user_idx"),
    explode(col("recommendations")).alias("rec")
).select(
    "user_idx",
    col("rec.product_idx").alias("product_idx"),
    col("rec.rating").alias("predicted_rating")
)

print("Raccomandazioni esplose:")
exploded_recs.show(10)

In [33]:
# ============================================
# Converti product_idx in product_id originale
# ============================================

# Create IndexToString per convertire product_idx -> product_id
product_converter = IndexToString(
    inputCol="product_idx",
    outputCol="product_id_original",
    labels=product_model.labels
)

exploded_recs_with_id = product_converter.transform(exploded_recs)

print("Raccomandazioni con product_id:")
exploded_recs_with_id.select("user_idx", "product_id_original", "predicted_rating").show(10)

NameError: name 'exploded_recs' is not defined

In [21]:
# ============================================
# Esempio: Raccomandazioni per un utente specifico
# ============================================

# Ottieni la lista degli utenti
all_users = ratings_indexed.select("user_idx").distinct().collect()
sample_user_idx = all_users[0]["user_idx"]

print("Utente campione (user_idx):", sample_user_idx)
print("\nProdotti che ha recensito (train set):")
train_data.filter(F.col("user_idx") == sample_user_idx).select("product_id", "rating").show(truncate=False)

print("\nRaccomandazioni per questo utente:")
user_recs.filter(F.col("user_idx") == sample_user_idx).show(truncate=False)

26/04/08 16:51:43 WARN DAGScheduler: Broadcasting large task binary with size 74.7 MiB
ERROR:root:KeyboardInterrupt while sending command.:>              (0 + 0) / 10]
Traceback (most recent call last):
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# ============================================
# Esempio: Raccomandazioni per un prodotto specifico
# ============================================

# Trova un prodotto popolare nel test set
sample_product = test_data.groupBy("product_idx").count().orderBy(F.desc("count")).first()
sample_product_idx = sample_product["product_idx"]

print("Prodotto campione (product_idx):", sample_product_idx)

# Trova utenti che hanno recensito questo prodotto
print("\nUtenti che hanno recensito questo prodotto:")
test_data.filter(F.col("product_idx") == sample_product_idx).select("user_idx", "rating").show(10, truncate=False)

---

<a id='section-4'></a>
## 4. Valutazione Approfondita

Analisi aggiuntive per verificare la qualita' del modello.

In [22]:
# ============================================
# Metriche di qualita': Precision@K
# ============================================

print("Calcolo Precision@K...")

# Converti predictions in un DataFrame per joins
pred_df = predictions.select("user_idx", "product_idx", "prediction").withColumnRenamed("prediction", "pred_rating")
test_df = test_data.select("user_idx", "product_idx", "rating")

# Unisci per vedere rating reali vs previsti
eval_df = pred_df.join(test_df, ["user_idx", "product_idx"], "inner")

# Conta accurate entro 1 punto
accurate = eval_df.filter(F.abs(F.col("rating") - F.col("pred_rating")) <= 1.0).count()
total = eval_df.count()

print("\nPrecision@1 (entro 1 punto):", round(accurate/total*100, 2), "%")
print("Totale valutazioni:", total)

# errore medio assoluto
mae = eval_df.select(F.avg(F.abs(F.col("rating") - F.col("pred_rating")))).first()[0]
print("\nMAE (Mean Absolute Error):", mae)

Calcolo Precision@K...


26/04/08 16:53:34 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/08 16:53:39 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/08 16:53:42 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/08 16:53:45 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
ERROR:root:KeyboardInterrupt while sending command.10][Stage 1454:>(0 + 0) / 10]
Traceback (most recent call last):
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [23]:
# ============================================
# Analisi residui
# ============================================

print("=== Analisi Residui (Errori di Predizione) ===")

eval_df.withColumn("error", F.col("rating") - F.col("pred_rating")).select("error").describe().show()

print("\nDistribuzione errori per range:")
eval_df.withColumn("error", F.col("rating") - F.col("pred_rating")).withColumn(
    "error_bucket",
    F.when(F.col("error") < -2, "<-2")
    .when(F.col("error") < -1, "-2 to -1")
    .when(F.col("error") < 0, "-1 to 0")
    .when(F.col("error") < 1, "0 to 1")
    .when(F.col("error") < 2, "1 to 2")
    .otherwise(">2")
).groupBy("error_bucket").count().orderBy("error_bucket").show(truncate=False)

=== Analisi Residui (Errori di Predizione) ===


26/04/08 16:54:52 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/anaconda3/envs/DataMining/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

26/04/08 16:54:56 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/08 16:55:00 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB
26/04/08 16:55:04 WARN DAGScheduler: Broadcasting large task binary with size 90.7 MiB


In [ ]:
# ============================================
# Valutazione per categoria di prodotto
# ============================================

# Unisci con nodes_df per ottenere la categoria
nodes_for_join = nodes_df.withColumnRenamed("id", "product_id")
nodes_for_join = nodes_for_join.withColumn("product_id", F.col("product_id").cast("integer"))

eval_with_group = eval_df.join(nodes_for_join.select("product_id", "group"), "product_id", "left")

print("=== RMSE per Gruppo ===")
eval_with_group.groupBy("group").agg(
    F.count("*").alias("count"),
    F.sqrt(F.mean(F.pow(F.col("rating") - F.col("pred_rating"), 2))).alias("rmse")
).orderBy("rmse").show(truncate=False)

---

## Conclusione

Questo notebook ha eseguito un'analisi completa su larga scala dei dati Amazon:

1. **Exploratory Data Analysis**: Analisi della struttura del grafo, distribuzioni dei gradi, statistiche dei ratings

2. **PageRank**: Identificazione dei prodotti piu' importanti nel grafo di similarita'

3. **ALS Recommender System**: Modello di collaborative filtering con matrix factorization
   - Training con 80% dei dati
   - RMSE sul test set: ~0.85-1.00 (buono per scala 1-5)
   - Generazione raccomandazioni per tutti gli utenti

4. **Valutazione**: Metriche RMSE, MAE, Precision@K, analisi residui per categoria

In [ ]:
# ============================================
# Pulizia
# ============================================

print("Ferrmando Spark session...")
spark.stop()
print("Spark session chiusa.")

Ferrmando Spark session...
Spark session chiusa.


26/04/20 15:36:12 ERROR Utils: Uncaught exception in thread Executor task launch worker for task 21.0 in stage 2974.0 (TID 2305)
java.lang.NullPointerException: Cannot invoke "org.apache.spark.SparkEnv.blockManager()" because the return value of "org.apache.spark.SparkEnv$.get()" is null
	at org.apache.spark.scheduler.Task.$anonfun$run$3(Task.scala:144)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1509)
	at org.apache.spark.scheduler.Task.run(Task.scala:142)
	at org.apache.spark.executor.Executor$TaskRunner.$anonfun$run$3(Executor.scala:554)
	at org.apache.spark.util.Utils$.tryWithSafeFinally(Utils.scala:1529)
	at org.apache.spark.executor.Executor$TaskRunner.run(Executor.scala:557)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at java.base/java.lang.Thread.run(Thread.java:840)
26/04/20 15:36:12 ERROR Utils: Uncaught exceptio